<a href="https://colab.research.google.com/github/jatinkumar-creator/machine-learning-portfolio/blob/main/04_Regression/notebooks/06_Regression_Cost_Functions_and_Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
#Generate a dataset
X, y = make_regression(
    n_samples=300,
    n_features=10,
    n_informative=5,
    noise=20,
    random_state=42
)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
#Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
#Build three regression models
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "Lasso Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=1.0, max_iter=10000))
    ])
}

In [ ]:
#Train and evaluate the models
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Train R2": r2_score(y_train, train_predictions),
        "Test R2": r2_score(y_test, test_predictions),
        "MAE": mean_absolute_error(y_test, test_predictions),
        "MSE": mean_squared_error(y_test, test_predictions),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, test_predictions)
        )
    })

results_df = pd.DataFrame(results)

results_df.round(3)

In [ ]:
#Compare model coefficients
coefficient_comparison = pd.DataFrame({
    name: model.named_steps["model"].coef_
    for name, model in models.items()
}, index=[f"Feature_{i}" for i in range(X.shape[1])])

coefficient_comparison.round(3)

In [ ]:
#Visualize coefficients
coefficient_comparison.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Coefficient Comparison: Linear, Ridge and Lasso")

plt.xlabel("Features")
plt.ylabel("Coefficient Value")

plt.xticks(rotation=45)
plt.axhline(y=0, linestyle="--")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#Experiment with different alpha values
alpha_values = [0.01, 0.1, 1, 10, 100, 1000]

ridge_results = []

for alpha in alpha_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    ridge_results.append({
        "Alpha": alpha,
        "Test R2": r2_score(y_test, predictions),
        "Test MAE": mean_absolute_error(y_test, predictions),
        "Test RMSE": np.sqrt(
            mean_squared_error(y_test, predictions)
        )
    })

ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df.round(3)

In [ ]:
#Visualize the effect of alpha
plt.plot(
    ridge_results_df["Alpha"],
    ridge_results_df["Test R2"],
    marker="o"
)

plt.xscale("log")

plt.xlabel("Alpha (log scale)")
plt.ylabel("Test R2")

plt.title("Effect of Regularization Strength on Ridge Regression")

plt.show()